In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/25 20:42:11 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/25 20:42:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bba2b4a8-0aea-4a16-a577-239383488615;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 67ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

In [2]:
from datetime import date

from pyspark.sql import functions as F

from foundry.pipeline import TrialBalancePipeline
from foundry.repository import TrialBalanceRepository
from foundry.config.settings import (
    CSV_TABLE_LOCATIONS, 
    POSTGRES_TABLE_LOCATIONS
)

from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from spec import SpecClient
from atlas import AtlasClient
from reference import ReferenceClient


def display_df(df):
    display(df.toPandas())

In [3]:
BUSINESS_DT = date(2025, 3, 31)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

spec = SpecClient.from_db(
    spark,
    transformation_table = 'spec.transformation',
    file_layout_table = 'spec.file_layout'
    
)
atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)
reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

pipeline = TrialBalancePipeline(
    business_dt=BUSINESS_DT,
    repository=repository,
    spec=spec,
    atlas=atlas,
    reference=reference,
    run_tracker=run_tracker,
)

In [4]:
pipeline_result = pipeline.execute()

print(f"Pipeline run complete: {pipeline_result}")

business_dt = pipeline._config.business_dt
batch_id = pipeline._config.batch_id

Pipeline run complete: PipelineResult(identity=RunIdentity(workflow_run_id=UUID('562f770f-6685-4887-8ad4-5f713e75496c'), run_id=UUID('f65a23cc-4ccf-4d73-ab85-f3740c002396'), parent_run_id=None), status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, zones=(ZoneResult(identity=RunIdentity(workflow_run_id=UUID('562f770f-6685-4887-8ad4-5f713e75496c'), run_id=UUID('d65f0bba-0f9f-4162-a1cc-0e65a13630a2'), parent_run_id=UUID('f65a23cc-4ccf-4d73-ab85-f3740c002396')), zone='STAGING', status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, record_count=96), ZoneResult(identity=RunIdentity(workflow_run_id=UUID('562f770f-6685-4887-8ad4-5f713e75496c'), run_id=UUID('82575f3e-4289-45ea-8fa8-8b1eec381a56'), parent_run_id=UUID('f65a23cc-4ccf-4d73-ab85-f3740c002396')), zone='ENRICHMENT', status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, record_count=16), ZoneResult(identity=RunIdentity(workflow_run_id=UUID('562f770f-6685-4887-8ad4-5f713e75496c'), run_id=UUID('b9e9ba66-c22d-471f-80ea-bdf9800182be'), parent_run_id=UUID('f65a23cc-4ccf-4

In [5]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,...,SRC_ACCT_CATEGORY,SRC_ACCT_TYPE,NORM_ACCT_SIGN,SRC_CLIENT_ID,SRC_CLIENT_NM,CPTY_REF_ID,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD
0,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,None,375545,src_prev_day_bal_amt,CAD,3424081.950000000000,CAD
1,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,None,375545,src_current_day_debit,CAD,0E-12,CAD
2,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,None,375545,src_current_day_credit,CAD,-709.880000000000,CAD
3,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,None,375545,src_current_day_eod_balance,CAD,3423372.070000000000,CAD
4,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,None,375545,src_back_valued_adjustment,CAD,0E-12,CAD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,None,None,None,src_current_day_debit,USD,0E-12,USD
92,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,None,None,None,src_current_day_credit,USD,0E-12,USD
93,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,None,None,None,src_current_day_eod_balance,USD,-27108.500000000000,USD
94,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,None,None,None,src_back_valued_adjustment,USD,0E-12,USD


In [6]:
stg_df = repository.read_staging(business_dt, batch_id)

display_df(stg_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,SRC_ENTITY_CD,...,POSTING_MEASURE_CCY_CD,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-1,NKC,...,CAD,PREVIOUS_DAY_BALANCE,REPORTABLE,CAD,3424081.950000000000,1.000000000000,3424081.950000000000,DEBIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2
1,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-2,NKC,...,CAD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,CAD,0E-12,1.000000000000,0E-12,DEBIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2
2,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-3,NKC,...,CAD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,CAD,-709.880000000000,1.000000000000,-709.880000000000,DEBIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2
3,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-4,NKC,...,CAD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,CAD,3423372.070000000000,1.000000000000,3423372.070000000000,DEBIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2
4,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-5,NKC,...,CAD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,CAD,0E-12,1.000000000000,0E-12,DEBIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-2-92,NSA,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2
92,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-2-93,NSA,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2
93,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-2-94,NSA,...,USD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,-27108.500000000000,1.000000000000,-27108.500000000000,CREDIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2
94,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-2-95,NSA,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT,562f770f-6685-4887-8ad4-5f713e75496c,d65f0bba-0f9f-4162-a1cc-0e65a13630a2


In [7]:
enr_df = repository.read_enrichment(business_dt, batch_id)

display_df(enr_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,ENRICHMENT_ID,...,GL_ACCOUNT_CR,GL_ACCOUNT_CR_DESC,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-6,ENR-250331-250331-2-1,...,207150,BANK OVERDRAFTS,100144,100011,000000,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
1,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-2,STG-250331-250331-2-12,ENR-250331-250331-2-2,...,300102,CAPITAL SURPLUS,300102,310000,505750,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
2,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-3,STG-250331-250331-2-18,ENR-250331-250331-2-3,...,400008,INT INC - BANK DEPOSITS,400008,400001,000000,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
3,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-4,STG-250331-250331-2-24,ENR-250331-250331-2-4,...,301009,RETAINED EARNINGS (SOURCE SYSTEMS),301009,999999,A99999,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
4,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-5,STG-250331-250331-2-30,ENR-250331-250331-2-5,...,207150,BANK OVERDRAFTS,100144,100011,000000,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
5,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-6,STG-250331-250331-2-36,ENR-250331-250331-2-6,...,198510,ACCR FEE REC - UNDERWRITING FEES,198510,130020,000000,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
6,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-7,STG-250331-250331-2-42,ENR-250331-250331-2-7,...,509893,CTRL - UNDERWRITING FEES,509893,411120,000000,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
7,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-8,STG-250331-250331-2-48,ENR-250331-250331-2-8,...,301009,RETAINED EARNINGS (SOURCE SYSTEMS),301009,999999,A99999,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
8,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-9,STG-250331-250331-2-54,ENR-250331-250331-2-9,...,207150,BANK OVERDRAFTS,100205,100010,000000,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56
9,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-10,STG-250331-250331-2-60,ENR-250331-250331-2-10,...,299935,CTRL - FIRM INVENTORY,299935,120014,000000,000000,LOCAL_GAAP,11392:001,562f770f-6685-4887-8ad4-5f713e75496c,82575f3e-4289-45ea-8fa8-8b1eec381a56


In [8]:
rpt_df = repository.read_reporting(business_dt, batch_id)

display_df(rpt_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,ENRICHMENT_ID,...,GL_ACCOUNT_CR,GL_ACCOUNT_CR_DESC,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_BOOK_CD,GL_PRODUCT_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-1,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be
1,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-2,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be
2,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-3,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be
3,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-4,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be
4,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-2-5,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-2-92,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be
92,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-2-93,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be
93,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-2-94,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be
94,2,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-2-95,,...,,,,,,,,,562f770f-6685-4887-8ad4-5f713e75496c,b9e9ba66-c22d-471f-80ea-bdf9800182be


In [16]:
pst_df = repository.read_posting(business_dt, batch_id)

display_df(pst_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,ENRICHMENT_ID,...,BACK_VALUE_ADJUSTED_BALANCE,ADJUSTED_BALANCE,POSTING_PREVIOUS_DAY_BALANCE,POSTING_CURRENT_DAY_DEBIT_BALANCE,POSTING_CURRENT_DAY_CREDIT_BALANCE,POSTING_CURRENT_DAY_EOD_BALANCE,POSTING_BACK_VALUE_ADJUSTED_BALANCE,POSTING_ADJUSTED_BALANCE,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-1-6,ENR-250331-250331-1-1,...,0E-12,3423372.070000000000,3424081.950000000000,0E-12,-709.880000000000,3423372.070000000000,0E-12,3423372.070000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
1,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-2,STG-250331-250331-1-12,ENR-250331-250331-1-2,...,0E-12,-27500000.000000000000,-27500000.000000000000,0E-12,0E-12,-27500000.000000000000,0E-12,-27500000.000000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
2,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-3,STG-250331-250331-1-18,ENR-250331-250331-1-3,...,0E-12,-155492.000000000000,-155492.000000000000,0E-12,0E-12,-155492.000000000000,0E-12,-155492.000000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
3,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-4,STG-250331-250331-1-24,ENR-250331-250331-1-4,...,0E-12,-985690.000000000000,-985690.000000000000,0E-12,0E-12,-985690.000000000000,0E-12,-985690.000000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
4,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-5,STG-250331-250331-1-30,ENR-250331-250331-1-5,...,0E-12,1778940.020000000000,1778940.020000000000,0E-12,0E-12,1778940.020000000000,0E-12,1778940.020000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
5,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-6,STG-250331-250331-1-36,ENR-250331-250331-1-6,...,0E-12,56250.000000000000,56250.000000000000,0E-12,0E-12,56250.000000000000,0E-12,56250.000000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
6,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-7,STG-250331-250331-1-42,ENR-250331-250331-1-7,...,0E-12,-73500.000000000000,-73500.000000000000,0E-12,0E-12,-73500.000000000000,0E-12,-73500.000000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
7,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-8,STG-250331-250331-1-48,ENR-250331-250331-1-8,...,0E-12,-6779438.000000000000,-6779438.000000000000,0E-12,0E-12,-6779438.000000000000,0E-12,-6779438.000000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
8,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-9,STG-250331-250331-1-54,ENR-250331-250331-1-9,...,0E-12,170131525.470000000000,5131525.470000000000,165000000.000000000000,0E-12,170131525.470000000000,0E-12,170131525.470000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34
9,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-10,STG-250331-250331-1-60,ENR-250331-250331-1-10,...,0E-12,0.090000000000,0.090000000000,0E-12,0E-12,0.090000000000,0E-12,0.090000000000,571aa2e7-a7d2-4b0d-b676-a7063f33bbdd,0d47a654-9288-4548-a8f5-e64cbbaa5d34


In [10]:
int_df = repository.read_interface(business_dt, batch_id)

display_df(int_df)

,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,TRANSACTION_NUMBER,LINE_NUMBER,DEFAULT_CURRENCY,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,...,FX_RATE,SRC_PREV_DAY_BAL_AMT,SRC_CRNT_DAY_DEBIT,SRC_CRNT_DAY_CREDIT,SRC_CRNT_DAY_EOD_BALANCE,SRC_BACK_VALUED_ADJUSTMENT,SRC_MEASURE_TRANS_AMT,SRC_ACCT_FUNC_AMT,AS_OF_DATE,EXTRACT_DATE
0,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NKC-250331-250331,1,CAD,509530,A05001,000000,100144,100011,...,1.000000000000,3424081.950000000000,0E-12,-709.880000000000,3423372.070000000000,0E-12,3423372.070000000000,15606936.270000000000,2025-03-31,2025-03-31
1,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NKC-250331-250331,2,CAD,509530,A05001,000000,300102,310000,...,1.000000000000,-27500000.000000000000,0E-12,0E-12,-27500000.000000000000,0E-12,-27500000.000000000000,-82500000.000000000000,2025-03-31,2025-03-31
2,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NKC-250331-250331,3,CAD,509530,A05001,000000,400008,400001,...,1.000000000000,-155492.000000000000,0E-12,0E-12,-155492.000000000000,0E-12,-155492.000000000000,-466476.000000000000,2025-03-31,2025-03-31
3,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NKC-250331-250331,4,CAD,509530,A05001,000000,301009,999999,...,1.000000000000,-985690.000000000000,0E-12,0E-12,-985690.000000000000,0E-12,-985690.000000000000,1475473459.860000000000,2025-03-31,2025-03-31
4,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NKC-250331-250331,5,CAD,509530,A05001,000000,100144,100011,...,1.000000000000,1778940.020000000000,0E-12,0E-12,1778940.020000000000,0E-12,1778940.020000000000,15606936.270000000000,2025-03-31,2025-03-31
5,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NKC-250331-250331,6,CAD,509530,A05001,000000,198510,130020,...,1.000000000000,56250.000000000000,0E-12,0E-12,56250.000000000000,0E-12,56250.000000000000,168750.000000000000,2025-03-31,2025-03-31
6,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NKC-250331-250331,7,CAD,509530,A05001,000000,509893,411120,...,1.000000000000,-73500.000000000000,0E-12,0E-12,-73500.000000000000,0E-12,-73500.000000000000,-220500.000000000000,2025-03-31,2025-03-31
7,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NKC-250331-250331,8,CAD,509530,A05001,000000,301009,999999,...,1.000000000000,-6779438.000000000000,0E-12,0E-12,-6779438.000000000000,0E-12,-6779438.000000000000,1475473459.860000000000,2025-03-31,2025-03-31
8,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NSA-250331-250331,8,USD,505890,A04001,000000,100205,100010,...,1.000000000000,5131525.470000000000,165000000.000000000000,0E-12,170131525.470000000000,0E-12,170131525.470000000000,510394576.410000000000,2025-03-31,2025-03-31
9,562f770f-6685-4887-8ad4-5f713e75496c,8d4f08eb-e8a5-42b7-bee6-5d95fdd5ca04,ITBG-2-NSA-250331-250331,1,USD,505890,A04001,000000,299935,120014,...,1.000000000000,0.090000000000,0E-12,0E-12,0.090000000000,0E-12,0.090000000000,0.270000000000,2025-03-31,2025-03-31


In [11]:
# spark.stop()

In [17]:
def to_csv(df, file_name):
    (
        df.write
        .mode('overwrite')
        .option('header', True)
        .csv(f'data/sample/{file_name}.csv')
    )

In [23]:
to_csv(int_df, 'interface')